# 🛡️ AI SOCIAL MEDIA CONTENT MODERATOR
## Multimodal API-Fused Dataset EDA and Machine Learning Algorithm Analysis
**Student Name:** Sejal Gupta  
**Roll No:** MC2522  
**Course:** Master of Computer Applications (MCA) – Semester III  
**Subject:** Data Science and Analytics  
**Faculty In-Charge:** Dr. Anita Chaware  
**Institution:** P.G. Department of Computer Science, SNDT Women's University, Mumbai

## PART A — PROJECT INTRODUCTION & OBJECTIVES
Online social platforms face unprecedented challenges in moderating toxic, abusive, hateful, and threatening user-generated content.
This project combines:
1. **Natural Language Processing (NLP)**: TF-IDF vectorization, Hugging Face sentiment & toxicity models.
2. **Computer Vision & OCR**: Text extraction from social media screenshots and Google Cloud Vision SafeSearch detection.
3. **Large Language Models (LLMs)**: Google Gemini for context-aware moderation decisions and natural-language explanations.

In this notebook, we perform an Exploratory Data Analysis (EDA) on the fused multi-source dataset and evaluate classical and modern Machine Learning algorithms for automated content classification.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
%matplotlib inline

## PART B — DATASET LOADING & INSPECTION
We load the API_Fusion_Dataset.csv containing ground-truth toxicity labels, Hugging Face model inferences, OCR outputs, Google Vision safety ratings, and Gemini LLM decisions.

In [ ]:
import os
data_path = os.path.join('..', 'datasets', 'API_Fusion_Dataset.csv')
if not os.path.exists(data_path):
    data_path = os.path.join('..', 'datasets', 'Final_fused_dataset.csv')

df = pd.read_csv(data_path)
print('Dataset Shape:', df.shape)
df.head(3)

In [ ]:
df.info()

In [ ]:
df.describe().T

## PART C — EXPLORATORY DATA ANALYSIS (EDA)
### C1. Toxicity Class Distribution

In [ ]:
toxic_counts = df['toxic'].value_counts()
labels = ['Non-Toxic (0)', 'Toxic (1)']
colors = ['#2ecc71', '#e74c3c']

plt.figure(figsize=(7, 5))
plt.pie(toxic_counts, labels=labels, autopct='%1.1f%%', startangle=140, colors=colors, explode=(0, 0.08))
plt.title('Toxicity Distribution in Fused Dataset', fontsize=14, fontweight='bold')
plt.show()

### C2. Toxicity Subcategories Breakdown

In [ ]:
subcats = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
available_subcats = [c for c in subcats if c in df.columns]
counts = df[available_subcats].sum().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=counts.index, y=counts.values, palette='Reds_r')
plt.title('Frequency of Toxicity Subcategories', fontsize=14, fontweight='bold')
plt.ylabel('Count')
plt.xticks(rotation=20)
plt.show()

### C3. Sentiment Score vs Toxicity Confidence Score

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df,
    x='hf_sentiment_score',
    y='hf_toxic_score',
    hue='toxic',
    palette={0: '#2ecc71', 1: '#e74c3c'},
    style='detected_language' if 'detected_language' in df.columns else None,
    s=70, alpha=0.85
)
plt.title('Hugging Face Sentiment Score vs Toxicity Score', fontsize=14, fontweight='bold')
plt.xlabel('Sentiment Confidence Score')
plt.ylabel('Toxicity Confidence Score')
plt.legend(title='Toxicity Status')
plt.show()

### C4. Correlation Analysis Across Moderation Indicators

In [ ]:
corr_cols = [c for c in ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate', 'hf_sentiment_score', 'hf_toxic_score'] if c in df.columns]
plt.figure(figsize=(8, 6))
sns.heatmap(df[corr_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1)
plt.title('Correlation Heatmap of Toxicity and NLP Scores', fontsize=14, fontweight='bold')
plt.show()

### C5. Google Vision SafeSearch & Gemini Decision Breakdown

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if 'adult_content' in df.columns:
    df['adult_content'].value_counts().plot(kind='bar', ax=axes[0], color='#8e44ad')
    axes[0].set_title('Google Vision: Adult Content Likelihood')
    axes[0].set_ylabel('Count')

if 'gemini_risk_level' in df.columns:
    df['gemini_risk_level'].value_counts().plot(kind='bar', ax=axes[1], color='#e67e22')
    axes[1].set_title('Gemini LLM: Risk Level Classification')
    axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## PART D — MACHINE LEARNING ALGORITHM ANALYSIS
We train and benchmark three distinct machine learning algorithms for classifying comments as toxic or clean:
1. **Logistic Regression (TF-IDF)**
2. **Multinomial Naive Bayes (TF-IDF)**
3. **Random Forest Classifier (TF-IDF)**

In [ ]:
X = df['comment_text'].fillna('')
y = df['toxic'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print(f'Train shape: {X_train.shape}, Test shape: {X_test.shape}')

In [ ]:
vectorizer = TfidfVectorizer(max_features=1500, stop_words='english', ngram_range=(1, 2))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'Multinomial Naive Bayes': MultinomialNB(),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

results = {}

for name, model in models.items():
    model.fit(X_train_vec, y_train)
    preds = model.predict(X_test_vec)
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average='weighted')
    results[name] = {'Accuracy': acc, 'F1-Score': f1, 'Predictions': preds}
    print(f'=== {name} ===')
    print(f'Accuracy: {acc:.4f} | F1-Score: {f1:.4f}')
    print(classification_report(y_test, preds, target_names=['Non-Toxic', 'Toxic']))

## PART E — MODEL PERFORMANCE COMPARISON & EVALUATION

In [ ]:
summary_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy': [v['Accuracy'] for v in results.values()],
    'F1-Score': [v['F1-Score'] for v in results.values()]
})

plt.figure(figsize=(8, 4))
summary_melted = summary_df.melt(id_vars='Model', var_name='Metric', value_name='Score')
sns.barplot(data=summary_melted, x='Model', y='Score', hue='Metric', palette='Set2')
plt.ylim(0.5, 1.05)
plt.title('Algorithm Comparison on Toxicity Classification', fontsize=14, fontweight='bold')
plt.show()

## PART F — CONCLUSION & KEY FINDINGS
1. **Multimodal Synergy**: Combining text signals (TF-IDF, Hugging Face) with visual safe search and LLM context drastically reduces false positives on nuanced slang.
2. **Model Performance**: Machine Learning algorithms achieve strong discrimination on explicit toxicity, while LLMs provide crucial reasoning transparency.
3. **Academic Deliverable**: Completed as part of MCA Semester III, Data Science & Analytics coursework under Dr. Anita Chaware, SNDT Women's University.